# Model Comparison — Home Credit

Compare multiple regression models predicting **AMT_CREDIT** (loan amount) using the same features and data pipeline as `linear_regression.ipynb`.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 2. Load & prepare data

Load `application_train.csv`, select target and features, drop missing values, and split 80/20 with `random_state=42`.

In [2]:
DATA_PATH = r"C:\Users\hchin\OneDrive\Desktop\ml_project\Machine Learning End to End Project\csv\application_train.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

target = "AMT_CREDIT"
feature_cols = [
    "AMT_INCOME_TOTAL",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CNT_CHILDREN",
    "DAYS_BIRTH",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]

model_df = df[[target] + feature_cols].copy()
rows_before = len(model_df)
model_df = model_df.dropna()
rows_after = len(model_df)

print(f"Rows before dropna: {rows_before:,}")
print(f"Rows after dropna:  {rows_after:,}")
print(f"Dropped rows:       {rows_before - rows_after:,}")

X = model_df[feature_cols]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train size: {len(X_train):,} samples")
print(f"Test size:  {len(X_test):,} samples")

Dataset shape: (307511, 123)
Rows before dropna: 307,511
Rows after dropna:  109,483
Dropped rows:       198,028
Train size: 87,586 samples
Test size:  21,897 samples


## 3. Train models

Train six regression models with reasonable default hyperparameters.

In [3]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
    "Elastic Net": ElasticNet(alpha=1.0, l1_ratio=0.5),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
}

fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    print(f"Fitted: {name}")

Fitted: Linear Regression
Fitted: Ridge
Fitted: Lasso
Fitted: Elastic Net
Fitted: Decision Tree
Fitted: Random Forest


## 4. Evaluate on test set

Compute **R²**, **MAE**, and **RMSE** for each model on the held-out test set.

In [4]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return r2, mae, rmse

results = []
for name, model in fitted_models.items():
    r2, mae, rmse = evaluate_model(model, X_test, y_test)
    results.append({"Model": name, "R²": r2, "MAE": mae, "RMSE": rmse})
    print(f"{name}:")
    print(f"  R²:   {r2:.4f}")
    print(f"  MAE:  {mae:,.2f}")
    print(f"  RMSE: {rmse:,.2f}")
    print()

Linear Regression:
  R²:   0.9745
  MAE:  50,491.57
  RMSE: 65,922.50

Ridge:
  R²:   0.9745
  MAE:  50,491.62
  RMSE: 65,922.50

Lasso:
  R²:   0.9745
  MAE:  50,491.94
  RMSE: 65,922.48

Elastic Net:
  R²:   0.9744
  MAE:  50,761.98
  RMSE: 66,049.44

Decision Tree:
  R²:   0.9857
  MAE:  30,245.74
  RMSE: 49,318.78

Random Forest:
  R²:   0.9872
  MAE:  29,503.57
  RMSE: 46,650.90



## 5. Final comparison table

All models ranked by **R²** (descending).

In [5]:
comparison_df = pd.DataFrame(results).sort_values("R²", ascending=False).reset_index(drop=True)

display_df = comparison_df.copy()
display_df["R²"] = display_df["R²"].map(lambda x: f"{x:.4f}")
display_df["MAE"] = display_df["MAE"].map(lambda x: f"{x:,.2f}")
display_df["RMSE"] = display_df["RMSE"].map(lambda x: f"{x:,.2f}")

print("Model Comparison (sorted by R² descending)")
print("=" * 60)
print(display_df.to_string(index=False))

comparison_df

Model Comparison (sorted by R² descending)
            Model     R²       MAE      RMSE
    Random Forest 0.9872 29,503.57 46,650.90
    Decision Tree 0.9857 30,245.74 49,318.78
            Lasso 0.9745 50,491.94 65,922.48
            Ridge 0.9745 50,491.62 65,922.50
Linear Regression 0.9745 50,491.57 65,922.50
      Elastic Net 0.9744 50,761.98 66,049.44


,Model,R²,MAE,RMSE
0,Random Forest,0.987238,29503.571826,46650.903249
1,Decision Tree,0.985736,30245.739926,49318.775456
2,Lasso,0.974516,50491.937280,65922.481355
3,Ridge,0.974516,50491.624071,65922.496853
4,Linear Regression,0.974516,50491.569910,65922.500818
5,Elastic Net,0.974418,50761.976542,66049.439585


## 6. Best model

The model with the highest **R²** on the test set is listed above. Linear and regularized linear models typically perform well when features are strongly correlated with loan amount; tree-based models may capture non-linear patterns but can overfit with limited depth settings.